In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_transaction_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateTransactionId string"
        #     ",SourceSystem string"
        #     ",PortfolioKey string"
        #     ",PortfolioName string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",RealEstateTransactionKey string"
        #     ",RealEstateCepalKey string"
        #     ",IsActivePortfolioFlag boolean"
        #     ",EligibleRealEstateFlag boolean"
        #     ",NonEligibilityReason string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateTransaction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== prproperty_eligibility ==================

        prproperty_eligibility_extra_filters = None

        if prproperty_eligibility_extra_filters is not None:
            prproperty_eligibility_extra_filters.append(date_filter)
            prproperty_eligibility_final_filters = " AND ".join(
                prproperty_eligibility_extra_filters
            )
        else:
            prproperty_eligibility_final_filters = date_filter

        prproperty_eligibility_selected_fields = [
            "key1",
            "key2",
            "portfolio_code",
            "active_portfolio",
            "cepal_key",
            "eligible_property",
            "non_eligibility_reason",
            "businessdate",
        ]

        prproperty_eligibility = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prproperty_eligibility",
            prproperty_eligibility_final_filters,
            prproperty_eligibility_selected_fields,
        )

        prproperty_eligibility_primary_key_fields = [
            coalesce(prproperty_eligibility.key1.cast("string"), lit("-1")),
            coalesce(prproperty_eligibility.key2.cast("string"), lit("-1")),
            lit("thesis"),
        ]

        prproperty_eligibility = prproperty_eligibility.withColumn(
            "RealEstateTransactionId",
            sha2(concat_ws(";", *prproperty_eligibility_primary_key_fields), 256),
        )

        prproperty_eligibility_portfolio_join_condition = (
            prproperty_eligibility["portfolio_code"] == portfolio["Prefix"]
        )

        target_table_thesis = prproperty_eligibility.join(
            portfolio,
            prproperty_eligibility_portfolio_join_condition,
            "left",
        )

        # ================== real_estate_transaction ==================

        real_estate_transaction = target_table_thesis.select(
            prproperty_eligibility.RealEstateTransactionId,
            lit("thesis").alias("SourceSystem"),
            prproperty_eligibility.portfolio_code.alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            concat_ws(
                ";",
                coalesce(col("key1").cast("string"), lit("-1")),
                coalesce(col("key2").cast("string"), lit("-1")),
            ).alias("RealEstateTransactionKey"),
            prproperty_eligibility.cepal_key.alias("RealEstateCepalKey"),
            prproperty_eligibility.active_portfolio.cast("boolean").alias(
                "IsActivePortfolioFlag"
            ),
            prproperty_eligibility.eligible_property.alias("EligibleRealEstateFlag"),
            prproperty_eligibility.non_eligibility_reason.alias("NonEligibilityReason"),
            prproperty_eligibility.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        real_estate_transaction.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_transaction_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)